<a href="https://colab.research.google.com/github/huc1atwit/Stock-Market-Analysis/blob/main/COMP_3125_Stock_Analysis_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyarrow scikit-learn
!pip install datasets pyarrow
!pip install datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error

In [ ]:
dataset = load_dataset("paperswithbacktest/Stocks-Daily-Price", split="train[:5%]")
df = pd.DataFrame(dataset)

print(df.head())

In [ ]:
print(df.columns)

In [ ]:
df = df.sort_values(['symbol', 'date'])

# Next day close price per stock
df['next_close'] = df.groupby('symbol')['close'].shift(-1)

# Direction target
df['price_up'] = (df['next_close'] > df['close']).astype(int)

# Drop missing values
df = df.dropna().reset_index(drop=True)

In [ ]:
features = ['open', 'high', 'low', 'close']

In [ ]:
X = df[features]
y = df['price_up']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_cls = LogisticRegression(max_iter=1000)
model_cls.fit(X_train, y_train)

pred_cls = model_cls.predict(X_test)

acc = accuracy_score(y_test, pred_cls)
print("Classification Accuracy:", acc)

In [ ]:
X = df[features]
y = df['next_close']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_reg = LinearRegression()
model_reg.fit(X_train, y_train)

pred_reg = model_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred_reg))
print("RMSE:", rmse)

In [ ]:
plt.figure()
plt.scatter(y_test, pred_reg)
plt.xlabel('Actual Next Close')
plt.ylabel('Predicted Next Close')
plt.title('Actual vs Predicted Stock Prices')

plt.savefig("actual_vs_predicted.png")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pick one stock
sample_ticker = df['symbol'].iloc[0]
sample = df[df['symbol'] == sample_ticker].tail(200)

plt.figure(figsize=(10,5))
plt.plot(sample['date'], sample['close'], label='Close Price')

plt.title(f'Stock Closing Price Over Time ({sample_ticker})')
plt.xlabel('Date')
plt.ylabel('Close Price')

step = 10
plt.xticks(sample['date'][::step], rotation=45)

plt.legend()
plt.tight_layout()

# Save for README
plt.savefig("stock_trend.png")

plt.show()

This project used real historical stock market data from the Hugging Face Stocks Daily Price Dataset to explore whether simple machine learning models can identify patterns in financial markets. By using daily Open, High, Low, and Close prices, we built two models: one to classify whether a stock would go up or down the next day, and another to predict the next day’s closing price.

The results show that while machine learning can capture some basic relationships in stock price movements, the predictions are limited due to the unpredictable nature of financial markets and the lack of external factors such as news, economic events, and investor behavior. Overall, this project demonstrates how real-world financial data can be applied to machine learning techniques, while also highlighting the challenges of making accurate predictions in complex systems like the stock market.